# Practical 03: Choosing a Model Honestly

**Overfitting, cross-validation, and defensible model selection**

**SCSE3040 Machine Learning Operations | Bennett University | Session 2026-27**

| Item | Detail |
|---|---|
| Lectures | L04-L05 |
| Course Outcome | CO2 |
| Duration | 120 minutes |
| GPU | not required |
| Marks | 10 |

## Aim

P02 separated training from testing. P03 strengthens the experiment:

```text
development data -> cross-validation -> model/configuration choice
final test data  -> one independent evaluation after selection
```

The objective is not to find the fanciest algorithm. It is to make a model-selection decision without quietly training the human experimenter on the final test set.


## Before you start

Run the notebook from top to bottom.

The same synthetic delivery-time dataset from P02 is used. The final test partition created below must remain outside the cross-validation and tuning steps.

If notebook state becomes confusing, restart the kernel, clear outputs, and rerun from the beginning.


In [ ]:
import sys
from pathlib import Path

print("Python :", sys.version.split()[0])
print("Program:", sys.executable)
print("Folder :", Path.cwd())

_missing = []
for _name in ["numpy", "pandas", "sklearn"]:
    try:
        __import__(_name)
    except ImportError:
        _missing.append(_name)

if _missing:
    print("STOP. Missing imports:", ", ".join(_missing))
else:
    print("All good. Continue.")


## Step 1: load the shared dataset and create one final holdout

We create the 80/20 split once.

From this point:

- `X_train, y_train` are the **development data** used for fitting, cross-validation, and model selection.
- `X_test, y_test` are the **final held-out data**.

The names retain the familiar scikit-learn convention, but the role of the first partition is broader than one fitting run.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error

DATA = Path("..") / "data" / "delivery_times.csv"

if not DATA.exists():
    raise FileNotFoundError(
        f"{DATA} was not found. Complete P01/P02 setup or restore the shared course data."
    )

orders = pd.read_csv(DATA)

FEATURES = [
    "distance_km",
    "prep_time_min",
    "traffic_level",
    "rain",
]
TARGET = "delivery_min"

X = orders[FEATURES]
y = orders[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

print("dataset :", orders.shape)
print("develop :", X_train.shape, y_train.shape)
print("final   :", X_test.shape, y_test.shape)


### Verify

Expected:

```text
dataset : (600, 5)
develop : (480, 4) (480,)
final   : (120, 4) (120,)
```

Do not use the final test partition to choose model families or hyperparameters in the later CV steps.


## Step 2: build a reusable train/test diagnostic

For teaching purposes, we first inspect training and test errors of several fixed models so that overfitting becomes visible.

This diagnostic comparison is **illustrative**. Once we begin selecting configurations, the notebook switches to development-only cross-validation and treats the final test partition as untouched.


In [ ]:
def score_both_ways(model, name):
    model.fit(X_train, y_train)

    train_mae = mean_absolute_error(
        y_train,
        model.predict(X_train),
    )
    test_mae = mean_absolute_error(
        y_test,
        model.predict(X_test),
    )
    gap = test_mae - train_mae

    result = {
        "model": name,
        "train": train_mae,
        "test": test_mae,
        "gap": gap,
    }

    print(
        f"{name:<28} "
        f"train {train_mae:5.2f}  "
        f"test {test_mae:5.2f}  "
        f"gap {gap:5.2f}"
    )

    return result


### Interpreting the gap

For MAE:

```text
gap = test MAE - train MAE
```

A large positive gap is evidence consistent with overfitting.

A small gap is **not** automatically evidence of a good model. Both errors could be similarly high.

A slightly negative gap is also possible if the held-out sample happens to be easier than the training sample.


## Step 3: Model 1, Linear Regression

This is the simple model carried forward from P02.


In [ ]:
linear = score_both_ways(
    LinearRegression(),
    "LinearRegression",
)


On the supplied dataset, training and test MAE are both near 2 minutes. The small negative gap is not suspicious by itself.

The synthetic dataset was generated from an approximately linear relationship, so Linear Regression has an important structural advantage here. Do not generalise this ranking to unrelated datasets.


## Step 4: Model 2, unrestricted Decision Tree

An unrestricted tree has enough capacity to partition the training observations very finely.

Run the next cell and compare its training error with its held-out error.


In [ ]:
tree_full = score_both_ways(
    DecisionTreeRegressor(random_state=42),
    "DecisionTree (unrestricted)",
)


### Stop and read the numbers

The training MAE should be essentially zero, while the held-out MAE is much larger.

This is the central overfitting demonstration:

```text
excellent fit to seen observations != excellent generalisation
```

The algorithm is not "an overfitting algorithm" by definition. Overfitting is behaviour observed under a particular data, capacity, and evaluation setting.


## Step 5: Model 3, depth-4 Decision Tree

Now reduce the tree's capacity.


In [ ]:
tree4 = score_both_ways(
    DecisionTreeRegressor(
        max_depth=4,
        random_state=42,
    ),
    "DecisionTree (depth 4)",
)


The depth-4 tree has a much smaller train/test gap than the unrestricted tree, but its test MAE is worse on this dataset.

That is an important lesson: **reducing the gap is not the same as improving the model**.

A model can become too constrained and underfit.


## Step 6: Model 4, Random Forest

A Random Forest averages predictions from multiple trees.

Ensembling often reduces variance relative to one unrestricted tree, but it does not make overfitting impossible.


In [ ]:
forest = score_both_ways(
    RandomForestRegressor(
        n_estimators=50,
        random_state=42,
    ),
    "RandomForest (50 trees)",
)


## Step 7: inspect the fixed-model diagnostic table


In [ ]:
diagnostic_table = pd.DataFrame(
    [linear, tree_full, tree4, forest]
).sort_values("test")

diagnostic_table.round(3)


The table is useful for seeing model behaviour, but from this point onward we stop using the final test set to make development decisions.

The next question is:

> How can we compare and tune models while preserving an independent final evaluation?

Answer: cross-validation on the development partition only.


## Step 8: development-only 5-fold cross-validation

Five-fold CV divides `X_train, y_train` into five folds.

Each development observation is:

- used for validation once;
- used for fitting in the other four folds.

The final `X_test, y_test` partition is not part of these folds.


In [ ]:
def cross_validate_model(model, name):
    scores = -cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="neg_mean_absolute_error",
    )

    print(
        f"{name:<28} "
        f"MAE {scores.mean():5.2f} "
        f"(+/- {scores.std():4.2f})"
    )

    return {
        "mean": float(scores.mean()),
        "std": float(scores.std()),
        "folds": scores,
    }


cv_linear = cross_validate_model(
    LinearRegression(),
    "LinearRegression",
)

cv_tree4 = cross_validate_model(
    DecisionTreeRegressor(
        max_depth=4,
        random_state=42,
    ),
    "DecisionTree (depth 4)",
)

cv_forest = cross_validate_model(
    RandomForestRegressor(
        n_estimators=50,
        random_state=42,
    ),
    "RandomForest (50 trees)",
)


### What does `+/-` mean?

It is the standard deviation of the five validation MAEs in this notebook.

It describes fold-to-fold variability. It is **not** a formal confidence interval, and overlapping `mean +/- std` ranges are not a statistical test proving two models are equivalent.

Also notice the minus sign before `cross_val_score`. scikit-learn uses a higher-is-better scoring convention, so error metrics are represented internally as negative scores.


## Step 9: tune Decision Tree depth without using the final test set

We now use validation evidence to select a hyperparameter.

This is model selection, so it belongs entirely inside the development partition.


In [ ]:
depths = [2, 3, 4, 6, 8, None]
scores_by_depth = {}

for depth in depths:
    candidate = DecisionTreeRegressor(
        max_depth=depth,
        random_state=42,
    )

    fold_mae = -cross_val_score(
        candidate,
        X_train,
        y_train,
        cv=5,
        scoring="neg_mean_absolute_error",
    )

    scores_by_depth[depth] = float(fold_mae.mean())

    print(
        f"max_depth={str(depth):<5} "
        f"MAE {fold_mae.mean():5.2f} "
        f"(+/- {fold_mae.std():4.2f})"
    )

best_depth = min(
    scores_by_depth,
    key=scores_by_depth.get,
)

print()
print("best development-CV depth:", best_depth)


Read the overall pattern, not merely the winning number.

Very shallow trees underfit. Increasing capacity improves validation MAE substantially. On this dataset the error reaches its minimum around an intermediate depth and changes only modestly near the high-capacity end.

Do not assume every real validation curve will be perfectly U-shaped. The observed curve depends on the data, folds, model, and random state.


## Step 10: select using development evidence, then evaluate once

For the supplied synthetic data, Linear Regression has the lowest development-only CV MAE among the fixed candidates above.

It is also operationally attractive here because it is small, fast, and directly inspectable.

Only after making that selection do we fit the selected model on all development observations and evaluate it once on the final test partition.


In [ ]:
selected_name = "LinearRegression"
selected_model = LinearRegression()

selected_model.fit(X_train, y_train)

final_predictions = selected_model.predict(X_test)
final_test_mae = mean_absolute_error(
    y_test,
    final_predictions,
)

print("selected model :", selected_name)
print(f"final test MAE : {final_test_mae:.2f} min")


The final test MAE is evidence from this experimental design, not a guarantee of production performance.

A deployment decision should consider predictive performance together with factors such as stability, interpretability, latency, memory, maintainability, and monitoring requirements.


---

# Your turn

Complete T1, T2, and T3 in order.

Do not rename the required variables. The self-check uses them.

The most important restriction is in T2 and T3:

> **Do not use `X_test` or `y_test` for model or hyperparameter selection.**


## Task T1: measure overfitting manually

Fit:

```python
DecisionTreeRegressor(
    max_depth=12,
    random_state=42,
)
```

on `X_train, y_train`.

Without using `score_both_ways`, calculate:

```python
T1_train_mae
T1_test_mae
T1_gap
```

where:

```text
T1_gap = T1_test_mae - T1_train_mae
```

This task is deliberately manual so you can see exactly what the helper encapsulated.


In [ ]:
deep_tree = DecisionTreeRegressor(
    max_depth=12,
    random_state=42,
)

# TODO: fit deep_tree on X_train, y_train

# TODO: calculate the two MAEs
T1_train_mae = None
T1_test_mae = None

# TODO: test MAE minus training MAE
T1_gap = None

print(
    "train:", T1_train_mae,
    "test:", T1_test_mae,
    "gap:", T1_gap,
)


## Task T2: tune Random Forest depth with development-only CV

Evaluate:

```python
T2_depths = [2, 4, 6, None]
```

using:

```python
RandomForestRegressor(
    n_estimators=50,
    max_depth=depth,
    random_state=42,
)
```

For each depth, compute 5-fold MAE using **only**:

```python
X_train, y_train
```

Create:

```python
T2_scores
T2_best_depth
```

`T2_scores` must map each depth to its mean positive CV MAE.

The final test set must not be used in this task.


In [ ]:
T2_depths = [2, 4, 6, None]

# TODO: replace with {depth: mean_cv_mae, ...}
T2_scores = None

# TODO: choose the key with the lowest MAE
T2_best_depth = None

print(T2_scores)
print("best:", T2_best_depth)


## Task T3: build a reusable CV comparison function

Implement:

```python
compare(models)
```

where `models` is a dictionary of `{name: estimator}`.

For each model:

1. run 5-fold cross-validation on `X_train, y_train`;
2. convert `neg_mean_absolute_error` to positive MAE;
3. store both the mean and standard deviation;
4. return the complete dictionary.

Required shape:

```python
{
    "linear": {
        "cv_mae_mean": ...,
        "cv_mae_std": ...
    },
    ...
}
```

Do not fit or score against `X_test, y_test` inside this function.


In [ ]:
def compare(models):
    # TODO: return a nested dictionary containing CV mean and std
    return None


T3_table = compare({
    "linear": LinearRegression(),
    "tree4": DecisionTreeRegressor(
        max_depth=4,
        random_state=42,
    ),
    "forest": RandomForestRegressor(
        n_estimators=50,
        random_state=42,
    ),
})

print(T3_table)


---

## Self-check

The checks validate the required outputs.

Passing the checks confirms the requested objects and calculations, but it does not replace your explanation of why the workflow protects the final test set.


In [ ]:
# ------------------------------------------------------------------
# SELF-CHECK
# ------------------------------------------------------------------

_results = []


def _check(label, fn):
    try:
        ok = bool(fn())
    except Exception:
        ok = False
    _results.append((label, ok))


_check(
    "T1 | depth-12 tree is trained",
    lambda: hasattr(deep_tree, "tree_"),
)

_check(
    "T1 | train and test MAE match deep_tree",
    lambda:
        abs(
            float(T1_train_mae)
            - mean_absolute_error(
                y_train,
                deep_tree.predict(X_train),
            )
        ) < 0.01
        and
        abs(
            float(T1_test_mae)
            - mean_absolute_error(
                y_test,
                deep_tree.predict(X_test),
            )
        ) < 0.01,
)

_check(
    "T1 | gap is test minus train and positive",
    lambda:
        abs(
            float(T1_gap)
            - (
                float(T1_test_mae)
                - float(T1_train_mae)
            )
        ) < 1e-6
        and float(T1_gap) > 0,
)

_check(
    "T2 | one forest-CV score exists per requested depth",
    lambda: set(T2_scores) == {2, 4, 6, None},
)

_check(
    "T2 | scores match development-only 5-fold CV",
    lambda: all(
        abs(
            float(T2_scores[d])
            - float(
                (
                    -cross_val_score(
                        RandomForestRegressor(
                            n_estimators=50,
                            max_depth=d,
                            random_state=42,
                        ),
                        X_train,
                        y_train,
                        cv=5,
                        scoring="neg_mean_absolute_error",
                    )
                ).mean()
            )
        ) < 0.01
        for d in [2, 4, 6, None]
    ),
)

_check(
    "T2 | best depth is the lowest-CV-MAE depth",
    lambda:
        T2_best_depth
        == min(T2_scores, key=T2_scores.get),
)

_check(
    "T3 | compare returns the requested model names and metrics",
    lambda:
        set(T3_table) == {"linear", "tree4", "forest"}
        and all(
            set(T3_table[name])
            == {"cv_mae_mean", "cv_mae_std"}
            for name in T3_table
        ),
)

_linear_cv_reference = -cross_val_score(
    LinearRegression(),
    X_train,
    y_train,
    cv=5,
    scoring="neg_mean_absolute_error",
)

_check(
    "T3 | linear CV mean and std match development-only CV",
    lambda:
        abs(
            float(T3_table["linear"]["cv_mae_mean"])
            - float(_linear_cv_reference.mean())
        ) < 0.01
        and
        abs(
            float(T3_table["linear"]["cv_mae_std"])
            - float(_linear_cv_reference.std())
        ) < 0.01,
)

print("=" * 72)
print("SELF-CHECK   Practical 03: Choosing a Model Honestly")
print("=" * 72)

for _label, _ok in _results:
    print(f"  [{'PASS' if _ok else 'FAIL'}]  {_label}")

print("-" * 72)

_passed = sum(1 for _, _ok in _results if _ok)
print(f"  {_passed} of {len(_results)} checks passed")
print("=" * 72)

if _passed == len(_results):
    print("Well done. Add the model-choice justification and save the notebook.")
else:
    print("Read the FAIL lines, fix those tasks, and run this cell again.")


## What to submit

1. This notebook, executed top to bottom with outputs visible.
2. A final markdown cell containing exactly three substantive sentences:
   - which model/configuration you would put in front of real customers;
   - why, using evaluation evidence **and** at least one operational advantage;
   - what trade-off you accept compared with an alternative.

Save the file as:

```text
P03_<your-roll-number>.ipynb
```

### Marking

| Component | Marks |
|---|---:|
| Walkthrough completed with outputs visible | 3 |
| T1: overfitting measured as a gap | 2 |
| T2: forest depth selected by development-only CV | 3 |
| T3: reusable CV comparison function | 2 |
| **Total** | **10** |

### Read more

- <https://scikit-learn.org/stable/modules/cross_validation.html>
- <https://scikit-learn.org/stable/modules/tree.html>
- <https://scikit-learn.org/stable/modules/ensemble.html>
- <https://scikit-learn.org/stable/auto_examples/model_selection/plot_underfitting_overfitting.html>
